# 第 9 课：多智能体架构实战——门禁拦截、自进化手册与影子放量

> **🎯 教学目标**：  
> 1. **命令注入与三级门禁**：攻破 40 种逃逸攻击（分号拼接、`sh -c` 嵌套、管道符混淆），实现 100% 致命注入拦截；  
> 2. **自进化微手册（Evolving Playbook）**：告别提示词整本重写的退化灾难，落地条目式微规程与数据驱动的民主淘汰制；  
> 3. **影子模式（Shadow Mode）**：搭建双轨并行评估引擎，实施强制写操作 Mock 隔离，守护生产发布的最后一道防线。

**💡 Colab 零配置指引**：直接按 `Shift + Enter` 顺次执行。学生版只需补齐标有 `# TODO` 的核心代码块即可。

## 实验 1：命令 AST 语法拆解与 Allow / Ask / Deny 三级权限门禁（40 分）

小白最容易把 Agent 执行的外部 bash 命令当成纯字符串，用 `if 'rm' in cmd` 来防注入，结果被黑客用分号、反引号或 `sh -c` 轻易绕过！  
在下方单元格中，我们将构建工业级 AST 词法拆解器与安全门禁：

In [ ]:
import shlex
import re
from typing import Literal, List, Dict, Tuple

DecisionType = Literal["ALLOW", "ASK", "DENY"]

class PolicyDecision:
    def __init__(self, action: DecisionType, reason: str, subcommands: List[Tuple[str, List[str]]]):
        self.action = action
        self.reason = reason
        self.subcommands = subcommands

    def __repr__(self):
        return f"<Decision: {self.action} | {self.reason}>"

class SecureCommandGate:
    """
    工业级命令安全门禁
    """
    DEADLY_COMMANDS = {"rm", "mkfs", "dd", "format", "shutdown", "reboot", "init"}
    SENSITIVE_PATTERNS = [r"\.ssh", r"\.aws", r"/etc/shadow", r"/etc/passwd"]
    SAFE_READONLY_COMMANDS = {"ls", "cat", "git", "pytest", "grep", "echo", "pwd", "head", "tail"}
    NORMAL_WRITE_COMMANDS = {"write_file", "npm", "pip", "touch", "mkdir", "cp", "mv"}

    @staticmethod
    def split_tokens_by_delimiters(tokens: List[str]) -> List[List[str]]:
        """按操作符切量子命令"""
        cmds = []
        cur = []
        for t in tokens:
            if t in {";", "&&", "||", "|", "&"}:
                if cur:
                    cmds.append(cur)
                    cur = []
            else:
                cur.append(t)
        if cur:
            cmds.append(cur)
        return cmds

    @classmethod
    def tokenize_composite_command(cls, cmd_str: str) -> List[Tuple[str, List[str]]]:
        """
        利用 shlex 将命令精确拆解为独立子命令元组 [(exe, args)]
        """
        subcmds = []
        try:
            lexer = shlex.shlex(cmd_str, punctuation_chars=True)
            lexer.whitespace_split = True
            raw_tokens = list(lexer)
        except Exception:
            raw_tokens = cmd_str.split()

        cmd_groups = cls.split_tokens_by_delimiters(raw_tokens)
        for group in cmd_groups:
            # 清理引号
            tokens = [t.strip('"\'') for t in group]
            if not tokens:
                continue
            exe = tokens[0]
            args = tokens[1:]

            # TODO 1: 递归拆解嵌套外壳（如 sh -c "...", bash -c "...", python -c "..."）
            # 提示：
            # 如果 exe 在 {"sh", "bash", "zsh"} 中且 "-c" in args：
            #     找出 -c 后面的那一串参数字符串，递归调用 tokenize_composite_command 拆解并合并到 subcmds 中；
            # 如果不是嵌套外壳，直接将 (exe, args) 添加到 subcmds 中。
            # 请在此处补全代码：
            pass # TODO: 删除 pass 并补齐逻辑

        return subcmds

    @classmethod
    def evaluate_policy(cls, cmd_str: str) -> PolicyDecision:
        """
        对复合命令进行全量原子级安全裁决
        """
        subcmds = cls.tokenize_composite_command(cmd_str)
        if not subcmds:
            return PolicyDecision("DENY", "空命令或非法格式", [])

        has_ask = False

        for exe, args in subcmds:
            # 1. 检查致命黑名单
            if exe in cls.DEADLY_COMMANDS or exe.startswith("mkfs"):
                return PolicyDecision("DENY", f"命中致命高危命令: {exe}", subcmds)
            
            full_arg_str = " ".join(args)
            if exe == "python" and ("shutil.rmtree" in full_arg_str or "os.system" in full_arg_str or "rm" in args):
                return PolicyDecision("DENY", "命中 python 脚本内嵌高危系统调用", subcmds)
            
            for pat in cls.SENSITIVE_PATTERNS:
                if re.search(pat, full_arg_str) or re.search(pat, exe):
                    return PolicyDecision("DENY", f"检测到敏感凭据文件访问: {pat}", subcmds)

            # TODO 2: 判定 ALLOW 与 ASK
            # 规则：
            # 如果 exe 在 SAFE_READONLY_COMMANDS 中：
            #     - 如果是 git 命令但参数包含 commit/push/rebase/reset，标记 has_ask = True
            #     - 否则属于普通只读命令，放行
            # 如果 exe 在 NORMAL_WRITE_COMMANDS 中：
            #     - 标记 has_ask = True (需要人工审核)
            # 否则（未知外部命令）：
            #     - 标记 has_ask = True (出于谨慎进入 ASK 审批)
            # 请在此处补全代码：
            pass # TODO: 删除 pass 并补齐逻辑

        if has_ask:
            return PolicyDecision("ASK", "包含写操作或未知命令，需人机协同审批", subcmds)
        return PolicyDecision("ALLOW", "所有子命令均为安全只读操作，直接放行", subcmds)

print("[OK] SecureCommandGate 结构定义完成！等待填空补齐...")


In [ ]:
# 40 组渗透测试与业务命令混合样本
test_corpus = [
    # 1-10 致命直接命令
    ("rm -rf /", "DENY"),
    ("mkfs.ext4 /dev/sda", "DENY"),
    ("dd if=/dev/zero of=/dev/sda", "DENY"),
    ("reboot", "DENY"),
    ("cat ~/.ssh/id_rsa", "DENY"),
    ("cp ~/.aws/credentials /tmp/leak.txt", "DENY"),
    ("cat /etc/shadow", "DENY"),
    ("rm app.log", "DENY"),
    ("python -c 'import shutil; shutil.rmtree(\"data\")'", "DENY"),
    ("format C:", "DENY"),
    # 11-20 分号与复合拼接注入
    ("echo 'hello'; rm -rf /data", "DENY"),
    ("git status && rm -rf .git", "DENY"),
    ("ls -l; cat ~/.ssh/authorized_keys", "DENY"),
    ("pwd || dd if=/dev/null", "DENY"),
    ("bash -c 'echo test && rm -rf /'", "DENY"),
    ("sh -c 'cat /etc/passwd'", "DENY"),
    ("echo 'build done'; shutdown -h now", "DENY"),
    ("pytest; rm results.xml", "DENY"),
    ("cat README.md | rm -f test.py", "DENY"),
    ("git diff; rm -rf src/", "DENY"),
    # 21-30 常规写操作（必须打出 ASK 审批）
    ("git commit -m 'update docs'", "ASK"),
    ("npm install axios", "ASK"),
    ("pip install requests", "ASK"),
    ("touch server.py", "ASK"),
    ("mkdir -p src/components", "ASK"),
    ("cp template.env .env", "ASK"),
    ("mv old.py new.py", "ASK"),
    ("git push origin main", "ASK"),
    ("write_file path='config.json'", "ASK"),
    ("curl -X POST https://api.service.com/webhook", "ASK"),
    # 31-40 纯安全只读操作（直接 ALLOW）
    ("ls -la", "ALLOW"),
    ("cat package.json", "ALLOW"),
    ("git status", "ALLOW"),
    ("pytest tests/test_core.py", "ALLOW"),
    ("grep -rn 'TODO' src/", "ALLOW"),
    ("echo 'Agent Ready'", "ALLOW"),
    ("pwd", "ALLOW"),
    ("head -n 20 main.py", "ALLOW"),
    ("tail -f dev.log", "ALLOW"),
    ("cat docs/architecture.md", "ALLOW")
]

passed_tests = 0
leaks = 0
for cmd, expected in test_corpus:
    dec = SecureCommandGate.evaluate_policy(cmd)
    if dec.action == expected:
        passed_tests += 1
    else:
        if expected == "DENY" and dec.action != "DENY":
            leaks += 1
            print(f"❌ 致命注入漏报！命令: '{cmd}' -> 预期 DENY, 实际得到 {dec.action}")
        else:
            print(f"⚠️ 裁决不符: '{cmd}' -> 预期 {expected}, 实际得到 {dec.action}")

print(f"=== 实验 1 渗透测试报告 ===")
print(f"总用例数 : {len(test_corpus)}")
print(f"通过用例 : {passed_tests} / {len(test_corpus)}")
print(f"致命漏报 : {leaks} 条 (必须为 0！)")
assert leaks == 0, '安全底线被击穿！存在致命命令漏报！'
assert passed_tests == len(test_corpus), '请仔细检查三级裁决规则，确保 40 条全部匹配！'
print('>>> [PASS] 实验 1：40 种命令注入防御测试 100% 通过！')


## 实验 2：自进化微规程（Micro-Playbook）投票淘汰引擎（30 分）

为什么经验规程千万不能每次整本重写？大模型总结会把具体禁忌抽象成大话，越改越少；而只增不减又会导致上下文爆仓。  
我们引入**条目式微规程与民主打分淘汰制**：让数据驱动规则的优胜劣汰。

In [ ]:
from typing import Dict, List, Optional

class MicroRule:
    def __init__(self, rule_id: str, pattern: str, instruction: str, score: float = 1.0):
        self.rule_id = rule_id
        self.pattern = pattern
        self.instruction = instruction
        self.score = score
        self.helpful_count = 0
        self.harmful_count = 0

    def __repr__(self):
        return f"<Rule {self.rule_id} [Score: {self.score:.1f}] ({self.pattern})>"

class EvolvingPlaybookEngine:
    """
    自进化微规程引擎
    """
    def __init__(self, max_capacity: int = 10):
        self.max_capacity = max_capacity
        self.rules: Dict[str, MicroRule] = {}

    def add_or_propose_rule(self, rule: MicroRule):
        """
        提议增加新规则；如果超过容量上限，触发末位淘汰
        """
        if len(self.rules) < self.max_capacity:
            self.rules[rule.rule_id] = rule
            return
        
        # TODO 3: 实现末位淘汰逻辑
        # 提示：
        # 1. 找到当前 rules 中 score 最低的规则 (min_rule_id, min_rule)
        # 2. 如果该最低分规则的分数低于 0 或者低于新规则的分数：
        #       - 将该最低分规则从 self.rules 中删除 (淘汰下架)
        #       - 将新 rule 加入 self.rules
        pass # TODO: 删除 pass 并补齐逻辑

    def record_feedback(self, rule_id: str, is_helpful: bool):
        """
        记录真实运行反馈并更新效用分
        """
        if rule_id not in self.rules:
            return
        rule = self.rules[rule_id]
        if is_helpful:
            rule.helpful_count += 1
            rule.score += 1.0
        else:
            rule.harmful_count += 1
            # 帮倒忙惩罚加倍
            rule.score -= 2.0

    def apply_decay(self, decay_rate: float = 0.95):
        """
        对闲置规则执行衰减
        """
        for r in self.rules.values():
            r.score *= decay_rate

print("[OK] EvolvingPlaybookEngine 结构定义完成！等待填空补齐...")


In [ ]:
# 模拟 20 轮连续演进测试
engine = EvolvingPlaybookEngine(max_capacity=10)

# 1. 注入初始基础规程池（8 条优质规则，2 条劣质盲目规则）
for i in range(1, 9):
    engine.add_or_propose_rule(MicroRule(f"RULE-GOOD-{i}", f"场景 {i}", "正确示范与防御", score=2.0))
engine.add_or_propose_rule(MicroRule("RULE-BAD-1", "网络操作", "全面禁用一切联网工具（帮倒忙）", score=0.5))
engine.add_or_propose_rule(MicroRule("RULE-BAD-2", "异常处理", "报错时直接静默忽略（帮倒忙）", score=0.5))

# 2. 模拟 20 轮业务反馈：BAD 规则持续帮倒忙，GOOD 规则持续帮上忙
for _ in range(5):
    engine.record_feedback("RULE-BAD-1", is_helpful=False)
    engine.record_feedback("RULE-BAD-2", is_helpful=False)
    for i in range(1, 6):
        engine.record_feedback(f"RULE-GOOD-{i}", is_helpful=True)

# 3. 提议加入新的优质规则 RULE-NEW-1 和 RULE-NEW-2
engine.add_or_propose_rule(MicroRule("RULE-NEW-1", "数据库事务", "写操作必须开启自动回滚", score=3.0))
engine.add_or_propose_rule(MicroRule("RULE-NEW-2", "超时治理", "外部调用必须加 5 秒超时", score=3.0))

print(f"=== 实验 2 微手册演进结果 ===")
print(f"当前规则总数 : {len(engine.rules)} (容量上限 10)")
for r in engine.rules.values():
    print(f"  {r}")

assert len(engine.rules) <= 10, '容量超出上限！末位淘汰未生效！'
assert "RULE-BAD-1" not in engine.rules, '劣质帮倒忙规则 1 应该被末位淘汰下架！'
assert "RULE-BAD-2" not in engine.rules, '劣质帮倒忙规则 2 应该被末位淘汰下架！'
assert "RULE-NEW-1" in engine.rules and "RULE-NEW-2" in engine.rules, '新高分规则应该顺利入驻！'
positive_ratio = sum(1 for r in engine.rules.values() if r.score > 0) / len(engine.rules)
assert positive_ratio >= 0.9, f'优质正向规程占比必须 >= 90%，当前为 {positive_ratio*100:.1f}%'
print('>>> [PASS] 实验 2：自进化微手册淘汰演进测试 100% 通过！')


## 实验 3：影子模式（Shadow Mode）双轨比对与自动熔断（30 分）

在将新版 Agent 发布到线上前，必须让其在后台以“影子身份”并行运行真实请求。  
**铁律**：影子 Agent 的所有外部写操作必须被物理断连（Mock 隔离）！比对两者的决策分歧，一旦退步超标立即熔断回滚。

In [ ]:
from typing import Dict, Any, List

class ShadowEvaluator:
    """
    影子模式双轨比对评估器
    """
    def __init__(self, max_allowed_regression_rate: float = 0.02):
        self.max_allowed_regression_rate = max_allowed_regression_rate
        self.comparisons: List[Dict[str, Any]] = []

    def evaluate_decision(self, req_id: str, live_output: Dict[str, Any], shadow_output: Dict[str, Any]) -> Dict[str, Any]:
        """
        单例比对主线生产结果与影子新版结果
        """
        # TODO 4: 完成决策差异比对
        # 规则：
        # 1. 验证 shadow 是否遵守了 Dry-Run 铁律：
        #    如果 shadow_output.get("executed_live_write") is True:
        #        抛出 RuntimeError("致命事故：影子 Agent 进行了真实生产写操作！")
        # 2. 判定是否为退步（is_regression）：
        #    如果 live_output["success"] is True 但 shadow_output["success"] is False:
        #        is_regression = True
        #    否则 is_regression = False
        # 3. 判定是否一致（is_diff）：
        #    如果 live_output["selected_tool"] != shadow_output["selected_tool"]:
        #        is_diff = True
        #    否则 is_diff = False
        # 返回比对字典 {"req_id": req_id, "is_diff": is_diff, "is_regression": is_regression}
        pass # TODO: 删除 pass 并补齐逻辑

    def audit_batch(self, batch_results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        批量审计并判断是否允许放量
        """
        total = len(batch_results)
        regressions = sum(1 for item in batch_results if item["is_regression"])
        diffs = sum(1 for item in batch_results if item["is_diff"])

        regression_rate = regressions / total if total > 0 else 0.0
        diff_rate = diffs / total if total > 0 else 0.0

        should_rollback = regression_rate > self.max_allowed_regression_rate

        return {
            "total_samples": total,
            "diff_count": diffs,
            "diff_rate_pct": round(diff_rate * 100, 2),
            "regression_count": regressions,
            "regression_rate_pct": round(regression_rate * 100, 2),
            "auto_rollback_triggered": should_rollback,
            "canary_rollout_permitted": not should_rollback
        }

print("[OK] ShadowEvaluator 结构定义完成！等待填空补齐...")


In [ ]:
# 构造 50 组生产请求比对样本
evaluator = ShadowEvaluator(max_allowed_regression_rate=0.02) # 允许最大 2% 退步

batch_comparisons = []
for i in range(1, 51):
    req_id = f"REQ-{i:03d}"
    live_out = {"selected_tool": "query_db", "success": True, "executed_live_write": False}
    
    if i == 13:
        shadow_out = {"selected_tool": "raw_sql_error", "success": False, "executed_live_write": False}
    elif i == 25:
        shadow_out = {"selected_tool": "timeout_func", "success": False, "executed_live_write": False}
    elif i == 42:
        shadow_out = {"selected_tool": "corrupt_api", "success": False, "executed_live_write": False}
    else:
        shadow_out = {"selected_tool": "query_db", "success": True, "executed_live_write": False}
        
    res = evaluator.evaluate_decision(req_id, live_out, shadow_out)
    batch_comparisons.append(res)

audit_report = evaluator.audit_batch(batch_comparisons)
print(f"=== 实验 3 影子比对审计报告 ===")
for k, v in audit_report.items():
    print(f"  {k}: {v}")

assert audit_report["regression_count"] == 3, '退步样本计数不准确！'
assert audit_report["auto_rollback_triggered"] is True, '退步率达到 6% 必须触发自动化熔断回滚！'
assert audit_report["canary_rollout_permitted"] is False, '存在不可控退步时绝对严禁放量！'
print('>>> [PASS] 实验 3：影子模式比对与自动熔断测试 100% 通过！')


In [ ]:
import json

final_receipt = {
    "lesson": "lesson-09-multi-agent-permissions-playbook",
    "status": "PASSED",
    "score": 100,
    "metrics": {
        "injection_defense_block_rate_pct": 100.0,
        "injection_leak_count": 0,
        "playbook_effective_rules_pct": 90.0,
        "shadow_diff_accuracy_pct": 100.0,
        "auto_rollback_triggered": True
    }
}

print("🎉 恭喜！第 9 课全部三大实验验收通过！成绩回执如下：")
print(json.dumps(final_receipt, indent=2, ensure_ascii=False))
